# Basics &mdash; Trees: Root, Height, Branching Factor, and the $b^n$ Bound

**Concept 13 of the Basics decomposition:** *Trees: Root, Height, Branching Factor, and the $b^n$ Bound*

A tree of height $n$ and branching factor $b$ has at most $b^n$ leaves.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Trees-And-The-Branching-Bound/Concept-Trees-And-The-Branching-Bound.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Basics/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


A **tree** starts at a designated **root**. The recursive definition has a base case
that is easy to miss: **the root may itself be a leaf**, giving a tree of **height 0
with no edges**.

Otherwise the height is $n\ge1$, the root has one or more children &mdash; their count is
the **branching factor** &mdash; and each child roots a **disjoint** subtree of height
$\le n-1$, with **at least one** of height exactly $n-1$. That last clause is what
makes the height exact rather than an upper bound.

The counting fact the book uses:

> **A tree of height $n$ and branching factor at most $b$ has at most $b^n$ leaves.**

That bound fixes the pumping constant $N = b^{|V|+1}$ in the **context-free** Pumping
Lemma (Chapter 11, Concept 18): make a parse tree tall enough and some nonterminal
must repeat on a root-to-leaf path.

## 2. Definitions

### Trees as nested tuples

In [ ]:
# a tree is (label, [children])
def leaf(x):            return (x, [])
def node(x, kids):      return (x, kids)

def height(t):
    return 0 if not t[1] else 1 + max(height(k) for k in t[1])

def leaves(t):
    return 1 if not t[1] else sum(leaves(k) for k in t[1])

def branching(t):
    return max([len(t[1])] + [branching(k) for k in t[1]]) if t[1] else 0

def full_tree(b, n, label='x'):
    if n == 0: return leaf(label)
    return node(label, [full_tree(b, n - 1, label) for _ in range(b)])

### Paths, and the repeated-label argument

In [ ]:
def paths(t, acc=()):
    acc = acc + (t[0],)
    if not t[1]:
        yield acc
    for k in t[1]:
        yield from paths(k, acc)

def repeated_on_some_path(t):
    for p in paths(t):
        seen = set()
        for i, x in enumerate(p):
            if x in seen: return p, x
            seen.add(x)
    return None

<!-- nav-strip -->

---

&larr;&nbsp;[Basics&nbsp;12.&nbsp;Function Signatures: Domain, Codomain, Range, Onto/Into, Total](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Function-Signatures/Concept-Function-Signatures.ipynb) &nbsp;&middot;&nbsp; [**Basics** index](https://github.com/ganeshutah/Jove/blob/master/Basics/README.md) &nbsp;&middot;&nbsp; [Ch1&nbsp;1.&nbsp;The Motivating Question: Can Computers Do Everything?](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Can-Computers-Do-Everything/Concept-Can-Computers-Do-Everything.ipynb)&nbsp;&rarr;

---

## 3. Tests

**The base case:** a lone root is a tree of height 0.

In [ ]:
t = leaf('r')
print("  tree :", t)
print("  height %d, leaves %d, branching factor %d"
      % (height(t), leaves(t), branching(t)))
assert height(t) == 0 and leaves(t) == 1 and branching(t) == 0
print("\nHeight 0, no edges -- and it is still a tree.")

**The bound:** height $n$, branching $b$ gives at most $b^n$ leaves.

In [ ]:
print("%-6s %-6s %-12s %s" % ("b", "n", "leaves", "b^n"))
for b in [2, 3]:
    for n in range(5):
        t = full_tree(b, n)
        print("%-6d %-6d %-12d %d" % (b, n, leaves(t), b ** n))
        assert leaves(t) == b ** n
        assert height(t) == n

It is an **upper** bound: unbalanced trees have fewer leaves.

In [ ]:
skew = node('r', [leaf('a'), node('b', [leaf('c'), leaf('d')])])
print("  height %d, branching %d, leaves %d, bound %d"
      % (height(skew), branching(skew), leaves(skew),
         branching(skew) ** height(skew)))
assert leaves(skew) <= branching(skew) ** height(skew)

**Turn it around:** enough leaves forces height.

In [ ]:
b = 2
for want in [2, 5, 9, 17]:
    import math
    need = math.ceil(math.log(want, b))
    print("  to have %2d leaves with branching %d, height must be >= %d"
          % (want, b, need))
    assert b ** need >= want
print("\nThat is the direction the CFL pumping lemma uses.")

**And a tall enough parse tree repeats a nonterminal on some path.**

In [ ]:
# |V| = 3 nonterminals; a path longer than 3 must repeat one
tall = node('S', [node('A', [node('B', [node('S', [leaf('a')])])])])
r = repeated_on_some_path(tall)
print("  path :", ' -> '.join(r[0]))
print("  repeated label :", r[1])
assert r is not None
print()
nonterms = 3
print("with |V| = %d nonterminals, any root-to-leaf path of more than %d"
      % (nonterms, nonterms))
print("labels must repeat one -- pigeonhole.  So choose N = b^(|V|+1):")
for b in [2, 3]:
    print("   b=%d, |V|=%d  ->  N = %d" % (b, nonterms, b ** (nonterms + 1)))
print()
print("A string that long forces a tall tree, a tall tree forces a repeat,")
print("and the repeat is what you pump.  Chapter 11, Concept 18.")

## 4. Exercises


1. Why must **at least one** subtree have height exactly $n-1$?
2. How many *nodes* (not leaves) does a full $b$-ary tree of height $n$ have?
3. Derive $N = b^{|V|+1}$ from the bound, carefully.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Basics/Concept-Trees-And-The-Branching-Bound')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')